# 🤖 Ensamble de Modelos  

---

## 🎯 Objetivos académicos
1. Comprender el concepto de aprendizaje en ensamble
2. Comprender el concepto de aprendizaje de potenciación del gradiente 
3. Aplicar modelos de Boosting en la tarea de regresión



## ¿Qué es un modelo de ensamble?  
Un **modelo de ensamble** combina múltiples modelos base (débilmente o fuertemente predictivos) para generar un modelo más robusto y preciso. En lugar de confiar en una sola predicción, se aprovecha la *sabiduría del grupo* 👥.  


###  Intuición del modelo  
Imagina que preguntas a varias personas la misma pregunta complicada 🤔.  
Aunque cada una pueda equivocarse, **al promediar sus respuestas obtienes un resultado más confiable**.  
👉 Esto mismo ocurre en los modelos de ensamble: la combinación de varios reduce errores individuales.  



### Ventajas de los modelos  
✨ Mayor precisión en predicciones  
🛡️ Menor riesgo de *overfitting* (dependiendo del método)  
🔍 Pueden capturar diferentes patrones en los datos  
⚡ Son versátiles: se aplican tanto a clasificación como regresión  


###  Ejemplos de modelos de ensamble  
- 🌲 **Bagging (Bootstrap Aggregating)**: como *Random Forest*  
- 🚀 **Boosting**: como *Gradient Boosting, AdaBoost, XGBoost, LightGBM*  
- 🧩 **Stacking**: combina distintos algoritmos (ej. árboles + regresión logística)  
- 🗳️ **Voting Classifier**: mayoría de votos entre modelos  


### 🚀 Potenciación de Gradiente (Boosting)  

---

#### ¿Qué es un modelo de ensamble?  
Un **modelo de ensamble** combina varios modelos base (llamados *weak learners*, usualmente árboles poco profundos 🌳) para formar un modelo más fuerte y preciso. En *boosting*, estos modelos se entrenan **de manera secuencial**, corrigiendo los errores del anterior.  



####  Intuición del modelo  
Imagina que un grupo de estudiantes resuelve un examen 📘.  
- El primero se equivoca en varias preguntas.  
- El segundo revisa y se enfoca en esas preguntas falladas.  
- El tercero hace lo mismo con los errores que quedan.  

👉 Al final, la combinación de todos mejora el resultado global.  
Eso es **boosting**: cada nuevo modelo se entrena para corregir los errores de los anteriores.  



####  Ventajas de los modelos  
✨ Alta precisión en comparación con modelos individuales.  
🛡️ Reduce el sesgo al enfocarse en errores previos.  
⚡ Flexible: puede usarse tanto en clasificación como regresión.  
📊 Permite ajustar la importancia de cada modelo mediante tasas de aprendizaje (*learning rate*).  



####  Ejemplos de modelos de ensamble con boosting  
- 🔥 **AdaBoost**: ajusta pesos de las observaciones mal clasificadas.  
- 🚀 **Gradient Boosting**: usa gradiente descendente para mejorar el ajuste.  
- ⚡ **XGBoost**: versión optimizada, muy rápida y eficiente.  
- 🌱 **LightGBM**: diseñado para grandes volúmenes de datos.  
- 🌀 **CatBoost**: muy efectivo con variables categóricas.  


##  Ejemplo práctico: Predicción del Precio de Autos - Rusty Bargain

---

Rusty Bargain es un servicio de coches usados que desea crear una app para estimar el valor de mercado de un vehículo.


### 🎯 Objetivos 

- Predecir el precio de un coche en euros.
- Comparar modelos diferentes: regresión lineal, árbol de decisión, bosque aleatorio y potenciación del gradiente (LightGBM).
- Implementar manualmente una regresión lineal con descenso por gradiente.
- Medir:
  - Precisión: RECM (RMSE)
  - Tiempo de entrenamiento
  - Velocidad de predicción


### Consideraciones

- La regresión lineal sirve como prueba de cordura.
- Potenciación del gradiente debe funcionar mejor que regresión lineal, si no, algo está mal.
- LightGBM y CatBoost manejan categóricas; XGBoost requiere OHE.
- Usa `%%time` para medir tiempos en Jupyter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

In [ ]:
df = pd.read_csv("https://practicum-content.s3.us-west-1.amazonaws.com/datasets/car_data.csv")

In [ ]:
df.head()

### EDA y Limpieza de datos

En esta celda realizamos una serie de pasos clave para preparar los datos antes de entrenar nuestros modelos:

1. **Filtrado de valores extremos (outliers):**
   - `Price` se restringe entre 100 y 15,000 euros para eliminar errores o valores atípicos.
   - `RegistrationYear` se limita entre 1950 y 2025 para asegurarse de que los años sean válidos.
   - `Power` (potencia del vehículo) se restringe entre 10 y 500 caballos de fuerza para evitar valores irreales.

2. **Eliminación de columnas irrelevantes:**
   - Se eliminan columnas que no aportan valor predictivo como:
     - `NumberOfPictures` (todos los valores son 0)
     - Fechas (`DateCrawled`, `DateCreated`, `LastSeen`)
     - `PostalCode` (identificador geográfico muy granular)

3. **Eliminación de filas con valores faltantes:**
   - `df.dropna()` descarta cualquier fila que contenga `NaN`, garantizando que el modelo no se entrene con datos incompletos.

4. **Codificación de variables categóricas:**
   - Se seleccionan las columnas categóricas y se transforman en variables dummy con `pd.get_dummies()`, usando `drop_first=True` para evitar multicolinealidad.

5. **Definición de variables predictoras y objetivo:**
   - `X`: todas las columnas excepto `'Price'`, que serán las características usadas para predecir.
   - `y`: la columna `'Price'`, que es nuestro objetivo.

6. **División del conjunto de datos:**
   - Se divide en conjunto de entrenamiento (`X_train`, `y_train`) y prueba (`X_test`, `y_test`) usando un 75% para entrenar y 25% para evaluar.


In [ ]:
df = df[df['Price'].between(100, 15000)]
df = df[df['RegistrationYear'].between(1950, 2025)]
df = df[df['Power'].between(10, 250)]

In [ ]:
df = df.drop(columns=['NumberOfPictures', 'DateCrawled', 'DateCreated', 'LastSeen', 'PostalCode'])
df = df.dropna()

In [ ]:
categorical = ['VehicleType', 'Gearbox', 'Model', 'FuelType', 'Brand', 'NotRepaired']
df = pd.get_dummies(df, columns=categorical, drop_first=True)

In [ ]:
df.info(), display(df.head())

In [ ]:
X = df.drop('Price', axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Baseline (Regresion lineal)

---

In [ ]:
%%time
lr = LinearRegression()
lr.fit(X_train, y_train)
preds_lr = lr.predict(X_test)
rmse_lr = root_mean_squared_error(y_test, preds_lr)
print(f"RMSE Linear Regression: {rmse_lr:.2f}")

### Random  forest

---

In [ ]:
%%time
forest = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)
forest.fit(X_train, y_train)
preds_forest = forest.predict(X_test)
rmse_forest = root_mean_squared_error(y_test, preds_forest)
print(f"RMSE Random Forest: {rmse_forest:.2f}")

### ⚡LightGBM

---

In [ ]:
%%time
lgb_train = lgb.Dataset(X_train, y_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'max_depth': 10,
    'verbose': -1
}

gbm = lgb.train(params, lgb_train, num_boost_round=100)
preds_lgb = gbm.predict(X_test)

rmse_lgb = root_mean_squared_error(y_test, preds_lgb)

print(f"RMSE LightGBM: {rmse_lgb:.2f}")

### ⚡XGBoost

---

In [ ]:
%%time

# Convertir datasets al formato de XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Parámetros de entrenamiento
params = {
    'objective': 'reg:squarederror',  # regresión
    'eval_metric': 'rmse',            # métrica de evaluación
    'eta': 0.1,                       # learning rate
    'max_depth': 10,                  # profundidad máxima
    'verbosity': 0
}

# Entrenamiento del modelo
xgb_model = xgb.train(params, dtrain, num_boost_round=100)

# Predicciones
preds_xgb = xgb_model.predict(dtest)

# Calcular RMSE
rmse_xgb = root_mean_squared_error(y_test, preds_xgb)

print(f"RMSE XGBoost: {rmse_xgb:.2f}")

### 📊 Comparación final de modelos

In [ ]:
modelos = ['LinearRegression',  'RandomForest', 'LightGBM', 'XGBoost']
rmses = [rmse_lr,  rmse_forest, rmse_lgb, rmse_xgb]
pd.DataFrame({'Modelo': modelos, 'RMSE': rmses}).sort_values(by='RMSE').round()

## Para cerrar 💬🤔

----

1. Explica en tus palabras los conceptos de:
    - Aprendizaje en ensamble
    - Potenciación del gradiente (*Boosting*) 
2. En este punto: ¿Tienes algun modelo preferido? 

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 14](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨